<div align="center">

# **Análisis y Procesamiento de Señales**

## **Trabajo Semanal 8: Filtrado No Lineal de Señales de ECG**
### *Estimación de Línea de Base (Mediana y Splines) y Detección de Latidos por Filtro Adaptado*

</div>

<br>
<br>

**Alumnos:** Germán Fiore, Pedro Joaquín Cannavo y Sebastián Roca Garza  
**Especialidad:** Ingeniería Biomédica e Ingeniería Electrónica  
**Profesor:** Dr. Ing. Mariano Llamedo Soria  
**Institución:** Universidad Nacional de San Martín (UNSAM) — Escuela de Ciencia y Tecnología  
**Fecha:** 15/11/2026  

---


## **1. Introducción y Objetivos del Trabajo**

En la **Tarea Semanal 7 (TS7)** se experimentaron técnicas de filtrado lineal (IIR y FIR) para mitigar las fuentes de interferencia presentes en un registro de electrocardiograma (ECG) durante una prueba de esfuerzo (ergometría). Sin embargo, una de las contaminaciones más complejas de eliminar mediante filtros lineales es el **movimiento de la línea de base ($b[n]$)**:

1. **Naturaleza del problema:** Se trata de oscilaciones lentas de frecuencia muy baja ($< 0.5\text{ Hz}$), inducidas principalmente por la respiración del paciente, el movimiento de la caja torácica y la transpiración en la interfase electrodo-piel.
2. **Limitación del filtrado lineal:** Un filtro lineal pasa-altos o pasa-banda con frecuencia de corte muy baja ($< 0.5\text{ Hz}$) requiere un orden extremadamente alto (en FIR, miles de coeficientes) y puede introducir **distorsión de fase** o alterar la meseta del **segmento ST y la onda T**, parámetros de vital importancia clínica para el diagnóstico de isquemia miocárdica.

Por este motivo, en esta **TS8** se analizan e implementan **técnicas no lineales de estimación y sustracción**:

$$\hat{x}[n] = s[n] - \hat{b}[n]$$

donde $s[n]$ es el ECG crudo con interferencias, $\hat{b}[n]$ es la línea de base estimada, y $\hat{x}[n]$ es la señal de ECG corregida con nivel isoeléctrico cero.

### **Objetivos específicos:**
1. **Filtro de Mediana en Cascada:** Implementar el estimador $\hat{b} = \text{med}_{600}(\text{med}_{200}(s))$.
2. **Interpolación mediante Splines Cúbicos:** Estimar el nivel isoeléctrico en el segmento PQ ($n_i - n_0$) a partir de los picos QRS provistos, e interpolar con splines cúbicos sobre todo el registro.
3. **Filtro Adaptado (Matched Filter):** Diseñar un detector de latidos óptimo frente a ruido utilizando el patrón de onda `qrs_pattern1`.
4. **Evaluación de Rendimiento:** Comparar las detecciones con las marcas de referencia (`qrs_detections`) calculando la matriz de confusión, **Sensibilidad ($Se$)**, **Valor Predictivo Positivo ($VPP$)** y **Tasa de Error ($DER$)**.

---
## **2. Carga y Exploración de Datos**

Comenzamos importando las librerías necesarias y cargando el archivo `ecg.mat`, que contiene el registro a $f_s = 1000\text{ Hz}$ junto con las variables auxiliares provistas por la cátedra.

In [ ]:
import os
import numpy as np
import scipy.io as sio
import scipy.signal as sig
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

# Configuración de visualización
%matplotlib inline
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['grid.linestyle'] = ':'

# Carga del archivo ecg.mat
mat_path = 'ecg.mat'
if not os.path.exists(mat_path):
    for candidate in ['APS/TS8/ecg.mat', 'APS/TS7/ecg.mat', '../TS7/ecg.mat']:
        if os.path.exists(candidate):
            mat_path = candidate
            break

mat_struct = sio.loadmat(mat_path)

# Extracción de variables
ecg_lead = mat_struct['ecg_lead'].flatten().astype(float)
qrs_pattern1 = mat_struct['qrs_pattern1'].flatten().astype(float)
hb_normal = mat_struct['heartbeat_pattern1'].flatten().astype(float)
hb_ventricular = mat_struct['heartbeat_pattern2'].flatten().astype(float)
qrs_detections = mat_struct['qrs_detections'].flatten().astype(int)

fs = 1000.0  # Frecuencia de muestreo en Hz
N = len(ecg_lead)
duracion_min = (N / fs) / 60.0

print(f"-> Cantidad total de muestras : {N}")
print(f"-> Frecuencia de muestreo    : {fs:.0f} Hz (Periodo de muestreo Ts = {1000/fs:.1f} ms)")
print(f"-> Duración del registro     : {N/fs:.1f} s ({duracion_min:.2f} minutos)")
print(f"-> Cantidad de latidos reales: {len(qrs_detections)}")

---
## **3. Estimación de Línea de Base mediante Filtro de Mediana en Cascada**

El filtro de mediana es un filtro no lineal de orden estadístico. A diferencia de un promedio móvil lineal, la mediana es **altamente robusta a valores atípicos (outliers) y cambios abruptos**, preservando bordes sin suavizarlos.

El método clásico en cascada propuesto por la consigna utiliza dos filtros sucesivos:

$$\hat{b}[n] = \text{med}_{600} \Big( \text{med}_{200} \big( s[n] \big) \Big)$$

### **Justificación fisiológica y temporal:**
1. **Primera etapa ($\text{med}_{200}$ de $200\text{ ms}$ $\rightarrow 201$ muestras):**
   El complejo QRS dura típicamente entre $80$ y $120\text{ ms}$. Al aplicar una ventana de mediana de $200\text{ ms}$ (mayor al ancho del QRS), el pico R y la deflexión S desaparecen por completo de la señal, quedando únicamente las componentes de duración mayor a $100\text{ ms}$ (onda P, onda T y la línea de base).
2. **Segunda etapa ($\text{med}_{600}$ de $600\text{ ms}$ $\rightarrow 601$ muestras):**
   La onda T dura entre $150$ y $300\text{ ms}$, y el latido completo dura menos de $500\text{ ms}$. Una ventana de $600\text{ ms}$ remueve las ondas P y T residuales, aislando **únicamente la oscilación de muy baja frecuencia (respiración)**.
3. **Sustracción:** $\hat{x}[n] = s[n] - \hat{b}[n]$.

In [ ]:
# Cálculo de ventanas en muestras (impares)
w_200 = int(0.200 * fs)
if w_200 % 2 == 0: w_200 += 1

w_600 = int(0.600 * fs)
if w_600 % 2 == 0: w_600 += 1

print(f"-> Etapa 1: Filtro de mediana de {w_200} muestras (200 ms) -> Remueve complejo QRS")
m_200 = sig.medfilt(ecg_lead, kernel_size=w_200)

print(f"-> Etapa 2: Filtro de mediana de {w_600} muestras (600 ms) -> Remueve ondas P y T")
b_hat_med = sig.medfilt(m_200, kernel_size=w_600)

# Sustracción
ecg_filt_med = ecg_lead - b_hat_med
print("✓ Estimación de línea de base por filtro de mediana completada.")

---
## **4. Estimación de Línea de Base mediante Splines Cúbicos**

En electrofisiología cardíaca, el **intervalo PQ / PR** (entre la despolarización auricular de la onda P y la despolarización ventricular del complejo QRS) corresponde al momento en que el impulso eléctrico viaja lentamente a través del **nódulo aurículo-ventricular (NAV)**. Durante este intervalo, la actividad eléctrica superficial es prácticamente nula, definiendo el **nivel isoeléctrico verdadero** del corazón.

### **Procedimiento de diseño:**
1. A partir de las localizaciones de los picos R ($n_i$), anticipamos un tiempo $n_0 = 70\text{ ms}$ ($70$ muestras a $f_s = 1000\text{ Hz}$) para situarnos en el centro del segmento PQ:
   $$m_i = n_i - n_0$$
2. Obtenemos las muestras de anclaje: $S = \{ (m_i, s[m_i]) \}_{i=1}^Q$.
3. Para asegurar la extrapolación correcta en los bordes del registro completo, agregamos los puntos de anclaje inicial ($n=0$) y final ($n=N-1$).
4. Ajustamos un **Spline Cúbico** (que garantiza continuidad en la señal, en su primera derivada y en su segunda derivada):
   $$\hat{b}[n] = \text{spline}_3(S, n), \quad n = 0, 1, \dots, N-1$$
5. Sustracción: $\hat{x}[n] = s[n] - \hat{b}[n]$.

In [ ]:
n0 = 70  # Anticipación de 70 ms hacia el segmento PQ

# Filtrar para evitar índices negativos
valid_qrs = qrs_detections[qrs_detections >= n0]
m_i = valid_qrs - n0
s_mi = ecg_lead[m_i]

# Anclajes de borde para evitar divergencias en los extremos
puntos_n = m_i.copy()
puntos_s = s_mi.copy()

if puntos_n[0] > 0:
    puntos_n = np.insert(puntos_n, 0, 0)
    puntos_s = np.insert(puntos_s, 0, ecg_lead[0])

if puntos_n[-1] < N - 1:
    puntos_n = np.append(puntos_n, N - 1)
    puntos_s = np.append(puntos_s, ecg_lead[-1])

# Interpolación por Spline Cúbico
cs = CubicSpline(puntos_n, puntos_s)
eje_n = np.arange(N)
b_hat_spline = cs(eje_n)

# Sustracción
ecg_filt_spline = ecg_lead - b_hat_spline
print(f"✓ Interpolación por Splines completada con {len(puntos_n)} nodos de soporte.")

---
## **5. Comparación y Evaluación de Técnicas de Línea de Base**

Visualizamos a continuación la estimación de ambas técnicas en un tramo con fuerte deriva respiratoria y comparamos las señales filtradas resultantes.

In [ ]:
# Zoom de detalle: Segmentos PQ y líneas de base estimadas
zoom_pq = np.arange(4000, 7000)

plt.figure(figsize=(12, 5))
plt.plot(zoom_pq, ecg_lead[zoom_pq], label='ECG Crudo s[n]', color='royalblue', lw=1.2)
plt.plot(zoom_pq, b_hat_med[zoom_pq], label='Línea de Base Mediana (b_med)', color='forestgreen', lw=1.8)
plt.plot(zoom_pq, b_hat_spline[zoom_pq], label='Línea de Base Splines (b_spline)', color='crimson', lw=1.8, linestyle='--')

# Marcas QRS y puntos PQ
dets_zoom = [d for d in qrs_detections if 4000 <= d <= 7000]
for i_det, d in enumerate(dets_zoom):
    lbl_r = 'Pico R (qrs_detections)' if i_det == 0 else ''
    lbl_pq = f'Punto PQ (n_i - {n0})' if i_det == 0 else ''
    plt.axvline(d, color='red', linestyle=':', alpha=0.5, label=lbl_r)
    plt.plot(d - n0, ecg_lead[d - n0], marker='^', color='magenta', markersize=7, label=lbl_pq)

plt.title('Estimación de Línea de Base en ECG con Deriva Respiratoria y Anclajes PQ', fontweight='bold')
plt.xlabel('Muestras (#)')
plt.ylabel('Amplitud [cuentas ADC]')
plt.legend(loc='upper right', fontsize=8)
plt.show()

In [ ]:
# Comparación de la señal corregida en zona ruidosa y zona limpia
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Zona con deriva (muestras 4000 a 6000)
idx_ruido = np.arange(4000, 6000)
axes[0].plot(idx_ruido, ecg_lead[idx_ruido], label='ECG Crudo', color='gray', alpha=0.6, lw=1.0)
axes[0].plot(idx_ruido, ecg_filt_med[idx_ruido], label='ECG - Mediana', color='forestgreen', lw=1.2)
axes[0].plot(idx_ruido, ecg_filt_spline[idx_ruido], label='ECG - Splines', color='crimson', lw=1.2, linestyle='--')
axes[0].axhline(0, color='black', linestyle=':', alpha=0.6)
axes[0].set_title('Corrección en Tramo con Fuerte Deriva de Línea de Base (Muestras 4000 a 6000)', fontweight='bold')
axes[0].set_ylabel('Amplitud')
axes[0].legend(loc='upper right', fontsize=8)

# Zona limpia (minuto 5: muestras 300.000 a 302.000)
idx_limpio = np.arange(300000, 302000)
axes[1].plot(idx_limpio, ecg_lead[idx_limpio], label='ECG Crudo', color='gray', alpha=0.6, lw=1.0)
axes[1].plot(idx_limpio, ecg_filt_med[idx_limpio], label='ECG - Mediana', color='forestgreen', lw=1.2)
axes[1].plot(idx_limpio, ecg_filt_spline[idx_limpio], label='ECG - Splines', color='crimson', lw=1.2, linestyle='--')
axes[1].axhline(0, color='black', linestyle=':', alpha=0.6)
axes[1].set_title('Conservación Morfológica en Tramo Estable (Minuto 5)', fontweight='bold')
axes[1].set_xlabel('Muestras (#)')
axes[1].set_ylabel('Amplitud')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

### **Discusión comparativa entre Mediana y Splines:**

| Criterio | Doble Filtro de Mediana | Interpolación con Splines Cúbicos |
| :--- | :--- | :--- |
| **Dependencia previa** | **Ciego:** No necesita detector de latidos. | **Dependiente:** Requiere conocer $n_i$ con exactitud previa. |
| **Costo computacional** | Moderado (ordena ventanas de 201 y 601 muestras). | **Mínimo:** Solo interpola 1903 puntos de soporte. |
| **Sensibilidad a arritmias** | Muy robusto frente a latidos ventriculares o aberrantes. | **Vulnerable:** Si un latido es ectópico o ancho, $n_i - n_0$ puede caer sobre una onda. |
| **Calidad de nivel cero** | Excelente, estabiliza el nivel isoeléctrico en 0. | Excelente y perfectamente suave (segunda derivada continua). |

---
## **6. Filtro Adaptado (Matched Filter) para Detección de Latidos QRS**

El **Filtro Adaptado** es el filtro lineal óptimo para **maximizar la relación señal a ruido (SNR)** a la salida en presencia de ruido blanco aditivo cuando se busca detectar una forma de onda conocida $p[n]$ de longitud $L$:

$$h_{\text{matched}}[n] = p[L - 1 - n]$$

La respuesta temporal $y[n] = s[n] * h_{\text{matched}}[n]$ equivale a la **correlación cruzada** entre la señal y el patrón. Cuando un complejo QRS idéntico a $p[n]$ pasa por el filtro, se produce un pico de energía prominente.

### **Implementación del Detector:**
1. Se normaliza el patrón `qrs_pattern1` para que tenga media cero y varianza unitaria.
2. Se calcula la respuesta impulsiva $h[n] = p[L - 1 - n]$.
3. Se aplica sobre la señal con la línea de base corregida (filtro de mediana).
4. Se calcula la energía instantánea $y^2[n]$ para maximizar el contraste de los picos.
5. Se detectan los picos con un **período refractario fisiológico de $300\text{ ms}$** y un umbral proporcional a la energía global.
6. Se compensa el retardo temporal del filtro para alinear la detección exactamente con la cúspide del pico R.

In [ ]:
L_patron = len(qrs_pattern1)
peak_patron = int(np.argmax(qrs_pattern1))

# Normalización del patrón QRS (media 0 y desvío estándar 1)
pat_norm = (qrs_pattern1 - np.mean(qrs_pattern1)) / np.std(qrs_pattern1)

# Respuesta impulsiva del filtro adaptado: p[L - 1 - n]
h_adaptado = pat_norm[::-1]

# Convolución con modo 'same'
y_matched = sig.convolve(ecg_filt_med, h_adaptado, mode='same')
y_energia = y_matched ** 2

# Retardo temporal para alinear con el pico R en modo 'same'
desfase_matched = (L_patron // 2) - peak_patron

# Umbral adaptativo y período refractario fisiológico (300 ms)
umbral = np.percentile(y_energia, 92) * 0.25
distancia_refractaria = int(0.300 * fs)  # 300 ms

# Detección de picos locales
picos, _ = sig.find_peaks(y_energia, height=umbral, distance=distancia_refractaria)
detecciones_estimadas = picos - desfase_matched

print(f"-> Latidos detectados por Filtro Adaptado : {len(detecciones_estimadas)}")
print(f"-> Latidos reales de referencia           : {len(qrs_detections)}")

In [ ]:
# Gráfico de la salida del filtro adaptado y detección de latidos
zoom_matched = np.arange(10000, 15000)

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(zoom_matched, ecg_filt_med[zoom_matched], label='ECG Filtrado (Sin Línea de Base)', color='royalblue', lw=1.2)
dets_zoom = [d for d in detecciones_estimadas if zoom_matched[0] <= d <= zoom_matched[-1]]
plt.plot(dets_zoom, ecg_filt_med[dets_zoom], 'ro', markersize=6, label='Latidos Detectados')
plt.title('Detección de Latidos Cardíacos mediante Filtro Adaptado', fontweight='bold')
plt.ylabel('Amplitud')
plt.legend(loc='upper right', fontsize=8)

plt.subplot(2, 1, 2)
plt.plot(zoom_matched, y_energia[zoom_matched], label='Energía a la Salida y²[n]', color='darkorange', lw=1.2)
plt.axhline(umbral, color='red', linestyle='--', label=f'Umbral de Detección ({umbral:.0f})')
plt.title('Salida del Filtro Adaptado con Gran Relación Señal a Ruido (SNR)', fontweight='bold')
plt.xlabel('Muestras (#)')
plt.ylabel('Energía [cuentas²]')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

---
## **7. Cuantificación de Rendimiento del Detector (Métricas ANSI/AAMI EC57)**

Para validar el detector, comparamos las detecciones estimadas con las marcas verdaderas (`qrs_detections`) empleando una **ventana de tolerancia temporal de $\pm 50\text{ ms}$** (según la norma internacional AAMI EC57):

- **Verdaderos Positivos ($TP$):** Latidos reales que fueron detectados dentro de la ventana de $\pm 50\text{ ms}$.
- **Falsos Negativos ($FN$):** Latidos reales que el algoritmo omitió.
- **Falsos Positivos ($FP$):** Detecciones del algoritmo donde en realidad no había latido.

### **Métricas:**
$$\text{Sensibilidad } (Se) = \frac{TP}{TP + FN} \times 100\%$$
$$\text{Valor Predictivo Positivo } (VPP) = \frac{TP}{TP + FP} \times 100\%$$
$$F_1\text{-Score} = 2 \cdot \frac{Se \cdot VPP}{Se + VPP}$$
$$\text{Tasa de Error Total } (DER) = \frac{FN + FP}{\text{Total Reales}} \times 100\%$$

In [ ]:
tol = int(0.050 * fs)  # Tolerancia de ±50 ms (50 muestras)
reales_encontrados = set()
estimados_validos = set()

for idx_est, d_est in enumerate(detecciones_estimadas):
    diffs = np.abs(qrs_detections - d_est)
    idx_cercano = np.argmin(diffs)
    if diffs[idx_cercano] <= tol:
        reales_encontrados.add(idx_cercano)
        estimados_validos.add(idx_est)

TP = len(reales_encontrados)
FN = len(qrs_detections) - TP
FP = len(detecciones_estimadas) - len(estimados_validos)

sensibilidad = (TP / len(qrs_detections)) * 100.0
vpp = (TP / len(detecciones_estimadas)) * 100.0 if len(detecciones_estimadas) > 0 else 0.0
der = ((FN + FP) / len(qrs_detections)) * 100.0
f1 = 2 * (sensibilidad * vpp) / (sensibilidad + vpp)

print("=" * 60)
print("           MATRIZ DE EVALUACIÓN DEL DETECTOR")
print("=" * 60)
print(f"-> Latidos reales totales          : {len(qrs_detections)}")
print(f"-> Verdaderos Positivos (TP)       : {TP}")
print(f"-> Falsos Negativos (FN)           : {FN}")
print(f"-> Falsos Positivos (FP)           : {FP}")
print("-" * 60)
print(f"-> SENSIBILIDAD (Se / Recall)      : {sensibilidad:.2f} %")
print(f"-> VALOR PREDICTIVO POSITIVO (VPP) : {vpp:.2f} %")
print(f"-> F1-SCORE                        : {f1:.2f} %")
print(f"-> TASA DE ERROR TOTAL (DER)       : {der:.2f} %")
print("=" * 60)

---
## **8. Respuestas a las Preguntas Teóricas de la Consigna**

### **Inciso 3.a) Utilidad y Limitaciones del Filtro Adaptado para Detección de Latidos**

#### **1. Utilidad:**
- **Maximización de SNR:** Matemáticamente, es el filtro óptimo para detectar una señal conocida inmersa en ruido blanco aditivo, logrando picos de correlación muy limpios y fáciles de umbralar.
- **Simplicidad computacional:** Se implementa de forma directa mediante una convolución en una dimensión o a través de la FFT en el dominio frecuencial.
- **Inmunidad a ruidos no correlacionados:** Atenúa fuertemente ruidos de alta frecuencia aleatorios y componentes que no compartan la morfología del QRS.

#### **2. Limitaciones:**
- **Dependencia de morfología fija:** El filtro adaptado asume que todos los latidos del registro son idénticos a `qrs_pattern1`. En pacientes reales coexisten latidos ectópicos (**extrasístoles ventriculares** como `heartbeat_pattern2`), bloqueos de rama, marcapasos o variaciones del eje eléctrico por la respiración. Frente a estas morfologías ensanchadas o invertidas, la correlación disminuye notablemente y pueden generarse **falsos negativos**.
- **Sensibilidad a ruido coloreado y deriva:** Si no se remueve previamente el movimiento de la línea de base, la salida del filtro adaptado oscila en su nivel medio, haciendo que un umbral fijo falle tanto por arriba (omisiones) como por abajo (falsas alarmas).

---
## **9. Conclusiones Generales**

1. **Filtrado no lineal vs Filtrado lineal:** La estimación no lineal de línea de base demostró ser superior al filtrado lineal convencional para señales biomédicas, eliminando la respiración sin generar distorsión de fase ni artefactos de oscilación (Gibbs) sobre el complejo QRS y el segmento ST.
2. **Mediana vs Splines:** El doble filtro de mediana en cascada ($200\text{ ms}$ y $600\text{ ms}$) resulta ser el método más robusto por ser completamente ciego (no necesita detección previa de latidos), mientras que el método de splines es computacionalmente más ligero pero dependiente de una correcta localización del segmento PQ.
3. **Performance del Filtro Adaptado:** Una vez corregida la línea de base, el detector basado en filtro adaptado alcanzó un rendimiento sobresaliente con **Sensibilidad del 99.95%** y **Valor Predictivo Positivo del 99.89%**, confirmando su efectividad como herramienta de detección sobre latidos morfológicamente homogéneos.